# 02 — 모델 정의 (Model Definition)

> 전처리 결과를 기반으로, 사용한 모델과 학습 설정을 명시적으로 정의합니다.

## 구성

1) 입력 데이터 구조 정의  
   - month_sin, month_cos, scenario_group 포함  

2) 모델 리스트 정의  
   - Ridge, Lasso, ElasticNet  
   - PLS  
   - RandomForest, CatBoost  
   - GAM  

3) 교차검증 전략 정의  
   - KFold  
   - LOGO (Leave-One-Group-Out)  

4) 하이퍼파라미터 설정  
5) 실험 설정 기록  

## 산출물

- model/models_config.json  
  → 모델/파라미터/검증 전략 설정 기록  

## 특징

- 학습은 수행하지 않음  
- 모델 구조와 실험 설정을 명확히 기록하는 단계  

In [1]:
# =========================
# 0) Imports & Paths
# =========================
import os, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

os.makedirs("model", exist_ok=True)

In [2]:
# =========================
# 1) Schema (from 01_preprocessing)
# =========================
FEATURES = [
    "Temp","Humid","CO2ppm",
    "TChl","Car",
    "Dio-RC","Eto-RC",
    "Fv-Fm",
    "Leaf_ExtractionYield","Root_ExtractionYield",
    "month_sin","month_cos"
]
TARGETS  = ["Leaf_TPC","Root_TPC","Leaf_TFC","Root_TFC"]
GROUPCOL = "scenario_group"  # LOGO 전용 그룹

schema = {"features": FEATURES, "targets": TARGETS, "group_column": GROUPCOL}
schema


{'features': ['Temp',
  'Humid',
  'CO2ppm',
  'TChl',
  'Car',
  'Dio-RC',
  'Eto-RC',
  'Fv-Fm',
  'Leaf_ExtractionYield',
  'Root_ExtractionYield',
  'month_sin',
  'month_cos'],
 'targets': ['Leaf_TPC', 'Root_TPC', 'Leaf_TFC', 'Root_TFC'],
 'group_column': 'scenario_group'}

## 2) 교차검증 스킴 (사용 코드 기반)
- **KFold(5, shuffle=True, random_state=42)** — 일반 성능 평가
- **LeaveOneGroupOut(LOGO)** — `scenario_group`별 일반화 성능 평가
- **GroupKFold(5)** — 월-시나리오 혼합 그룹(`scenario_group`×월 구간) 기반 평가(일부 CatBoost 실험에서 사용)

In [3]:
CV_SCHEMES = {
    "KFold": {
        "n_splits": 5,
        "shuffle": True,
        "random_state": 42
    },
    "LOGO": {
        "group_column": GROUPCOL
    },
    "GroupKFold": {
        "n_splits": 5,
        "grouping_note": "scenario_group × month_bin (H1/Summer/H2)"
    }
}
CV_SCHEMES

{'KFold': {'n_splits': 5, 'shuffle': True, 'random_state': 42},
 'LOGO': {'group_column': 'scenario_group'},
 'GroupKFold': {'n_splits': 5,
  'grouping_note': 'scenario_group × month_bin (H1/Summer/H2)'}}

## 3) 알고리즘 & 기본 파라미터 (사용 코드 그대로)
### 선형/기저 모델
- Ridge(alpha=1.0, random_state=42)
- Lasso(alpha=0.001, max_iter=10000, random_state=42)
- ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)
- PLSRegression(n_components=5)

### 트리 계열 (멀티타깃은 MultiOutput으로 래핑)
- RandomForestRegressor(n_estimators=600, max_depth=None, min_samples_leaf=2, max_features='sqrt', random_state=42)
- XGBRegressor(n_estimators=500, learning_rate=0.08, max_depth=5, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0, objective='reg:squarederror', random_state=42)
- CatBoostRegressor(iterations=800, depth=6, learning_rate=0.07, l2_leaf_reg=3.0, loss_function='RMSE', random_seed=42, verbose=False)

### GAM (해석용)
- LinearGAM: `s(i, n_splines, lam)`를 모든 피처에 합산
- 탐색 그리드 예시: lam ∈ logspace(1e-2, 10), n_splines ∈ {6,8,10,12}


In [4]:
MODELS_BASE = {
    "Ridge":      {"alpha": 1.0, "random_state": 42},
    "Lasso":      {"alpha": 0.001, "l1_ratio": None, "max_iter": 10000, "random_state": 42},
    "ElasticNet": {"alpha": 0.001, "l1_ratio": 0.5, "max_iter": 10000, "random_state": 42},
    "PLS":        {"n_components": 5},
    "RandomForest": {
        "n_estimators": 600, "max_depth": None, "min_samples_leaf": 2,
        "max_features": "sqrt", "random_state": 42
    },
    "XGBoost": {
        "n_estimators": 500, "learning_rate": 0.08, "max_depth": 5,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 2.0,
        "objective": "reg:squarederror", "random_state": 42
    },
    "CatBoost": {
        "iterations": 800, "depth": 6, "learning_rate": 0.07, "l2_leaf_reg": 3.0,
        "loss_function": "RMSE", "random_seed": 42, "verbose": False
    },
    "GAM": { "notes": "LinearGAM with per-feature s(i, n_splines, lam)" }
}
MODELS_BASE

{'Ridge': {'alpha': 1.0, 'random_state': 42},
 'Lasso': {'alpha': 0.001,
  'l1_ratio': None,
  'max_iter': 10000,
  'random_state': 42},
 'ElasticNet': {'alpha': 0.001,
  'l1_ratio': 0.5,
  'max_iter': 10000,
  'random_state': 42},
 'PLS': {'n_components': 5},
 'RandomForest': {'n_estimators': 600,
  'max_depth': None,
  'min_samples_leaf': 2,
  'max_features': 'sqrt',
  'random_state': 42},
 'XGBoost': {'n_estimators': 500,
  'learning_rate': 0.08,
  'max_depth': 5,
  'subsample': 0.9,
  'colsample_bytree': 0.9,
  'reg_lambda': 2.0,
  'objective': 'reg:squarederror',
  'random_state': 42},
 'CatBoost': {'iterations': 800,
  'depth': 6,
  'learning_rate': 0.07,
  'l2_leaf_reg': 3.0,
  'loss_function': 'RMSE',
  'random_seed': 42,
  'verbose': False},
 'GAM': {'notes': 'LinearGAM with per-feature s(i, n_splines, lam)'}}

## 4) 하이퍼파라미터 탐색 공간 (사용 코드 기반)
### CatBoost — 무작위/그리드 탐색에서 사용한 공간(대표)
- `bootstrap_type ∈ {Bayesian, Bernoulli}`
- `bagging_temperature ∈ {0, 0.25, 0.5, 1, 2}` (Bayesian일 때)
- `subsample ∈ {0.7, 0.8, 0.9, 1.0}` (Bernoulli일 때)
- `learning_rate ∈ {0.02, 0.03, 0.05, 0.07, 0.1}`
- `depth ∈ {4,5,6,7,8,9}`
- `l2_leaf_reg ∈ {3,5,7,10,20}`
- `colsample_bylevel ∈ {0.7,0.8,0.9,1.0}`
- `random_strength ∈ {0.5,1,2,3,5}`
- `iterations ∈ {2000,3000,5000}`

### GAM — 탐색 범위(예시)
- `lam ∈ logspace(1e-2, 10)`
- `n_splines ∈ {6,8,10,12}`


In [5]:
SEARCH_SPACES = {
    "CatBoost_random_or_grid": {
        "bootstrap_type": ["Bayesian","Bernoulli"],
        "bagging_temperature (if Bayesian)": [0, 0.25, 0.5, 1, 2],
        "subsample (if Bernoulli)": [0.7, 0.8, 0.9, 1.0],
        "learning_rate": [0.02, 0.03, 0.05, 0.07, 0.1],
        "depth": [4,5,6,7,8,9],
        "l2_leaf_reg": [3,5,7,10,20],
        "colsample_bylevel": [0.7,0.8,0.9,1.0],
        "random_strength": [0.5,1,2,3,5],
        "iterations": [2000,3000,5000]
    },
    "GAM": {
        "lam": "np.logspace(-2, 1, 8)  # 사용 예시",
        "n_splines": [6,8,10,12]
    }
}
SEARCH_SPACES

{'CatBoost_random_or_grid': {'bootstrap_type': ['Bayesian', 'Bernoulli'],
  'bagging_temperature (if Bayesian)': [0, 0.25, 0.5, 1, 2],
  'subsample (if Bernoulli)': [0.7, 0.8, 0.9, 1.0],
  'learning_rate': [0.02, 0.03, 0.05, 0.07, 0.1],
  'depth': [4, 5, 6, 7, 8, 9],
  'l2_leaf_reg': [3, 5, 7, 10, 20],
  'colsample_bylevel': [0.7, 0.8, 0.9, 1.0],
  'random_strength': [0.5, 1, 2, 3, 5],
  'iterations': [2000, 3000, 5000]},
 'GAM': {'lam': 'np.logspace(-2, 1, 8)  # 사용 예시', 'n_splines': [6, 8, 10, 12]}}

## 5) 실험 메모 (사용 코드에서 관찰된 예시 결과)
- **GAM(KFold)** 상위 설정 예시: lam=10.0, n_splines=6 → KF_R2≈0.84, LOGO_R2≈0.041 *(예시 수치)*
- **RandomForest**: KFold R2≈0.991, LOGO R2≈0.127 *(예시 수치)*
- **XGBoost**: KFold R2≈0.992, LOGO R2≈0.051 *(예시 수치)*
- **CatBoost**(기본): KFold R2≈0.995, LOGO R2≈0.180 *(예시 수치)*
- **CatBoost 튜닝**: Bayesian 샘플 중 KFold 상위 trial 예: `learning_rate=0.02, depth=7, l2_leaf_reg=3, colsample_bylevel=0.7, random_strength=2, iterations=5000, bagging_temperature=0.25` (LOGO 평균 R2≈0.239)

※ 위 수치는 *설명 기록용*으로 남기며, 재현은 03 노트북에서 수행합니다.

In [6]:
# =========================
# 6) Save definition → model/models_config.json
# =========================
config = {
    "schema": {
        "features": FEATURES,
        "targets": TARGETS,
        "group_column": GROUPCOL
    },
    "cv_schemes": {
        "KFold": {"n_splits": 5, "shuffle": True, "random_state": 42},
        "LOGO": {"group_column": GROUPCOL},
        "GroupKFold": {"n_splits": 5, "grouping_note": "scenario_group × month_bin (H1/Summer/H2)"}
    },
    "models": {
        "Ridge": {"alpha": 1.0, "random_state": 42},
        "Lasso": {"alpha": 0.001, "max_iter": 10000, "random_state": 42},
        "ElasticNet": {"alpha": 0.001, "l1_ratio": 0.5, "max_iter": 10000, "random_state": 42},
        "PLS": {"n_components": 5},
        "RandomForest": {"n_estimators": 600, "max_depth": None, "min_samples_leaf": 2, "max_features": "sqrt", "random_state": 42},
        "XGBoost": {"n_estimators": 500, "learning_rate": 0.08, "max_depth": 5, "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 2.0, "objective": "reg:squarederror", "random_state": 42},
        "CatBoost": {"iterations": 800, "depth": 6, "learning_rate": 0.07, "l2_leaf_reg": 3.0, "loss_function": "RMSE", "random_seed": 42, "verbose": False},
        "GAM": {"notes": "LinearGAM with per-feature s(i, n_splines, lam)"}
    },
    "search_spaces": {
        "CatBoost_random_or_grid": {
            "bootstrap_type": ["Bayesian","Bernoulli"],
            "bagging_temperature (if Bayesian)": [0, 0.25, 0.5, 1, 2],
            "subsample (if Bernoulli)": [0.7, 0.8, 0.9, 1.0],
            "learning_rate": [0.02, 0.03, 0.05, 0.07, 0.1],
            "depth": [4,5,6,7,8,9],
            "l2_leaf_reg": [3,5,7,10,20],
            "colsample_bylevel": [0.7,0.8,0.9,1.0],
            "random_strength": [0.5,1,2,3,5],
            "iterations": [2000,3000,5000]
        },
        "GAM": {"lam": "np.logspace(-2, 1, 8)", "n_splines": [6,8,10,12]}
    },
    "notes": {
        "catboost_feature_sets": [
            "all",
            "no_month (drop month_sin, month_cos)",
            "no_yield (drop Leaf_/Root_ExtractionYield)",
            "no_month_yield (drop both)"
        ],
        "priority": "LOGO generalization prioritized in later tuning",
        "repro": "Use 03_training_evaluation.ipynb for CV runs and model export"
    }
}

with open("model/models_config.json", "w", encoding="utf-8") as f:
    import json
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved → model/models_config.json")
config

Saved → model/models_config.json


{'schema': {'features': ['Temp',
   'Humid',
   'CO2ppm',
   'TChl',
   'Car',
   'Dio-RC',
   'Eto-RC',
   'Fv-Fm',
   'Leaf_ExtractionYield',
   'Root_ExtractionYield',
   'month_sin',
   'month_cos'],
  'targets': ['Leaf_TPC', 'Root_TPC', 'Leaf_TFC', 'Root_TFC'],
  'group_column': 'scenario_group'},
 'cv_schemes': {'KFold': {'n_splits': 5, 'shuffle': True, 'random_state': 42},
  'LOGO': {'group_column': 'scenario_group'},
  'GroupKFold': {'n_splits': 5,
   'grouping_note': 'scenario_group × month_bin (H1/Summer/H2)'}},
 'models': {'Ridge': {'alpha': 1.0, 'random_state': 42},
  'Lasso': {'alpha': 0.001, 'max_iter': 10000, 'random_state': 42},
  'ElasticNet': {'alpha': 0.001,
   'l1_ratio': 0.5,
   'max_iter': 10000,
   'random_state': 42},
  'PLS': {'n_components': 5},
  'RandomForest': {'n_estimators': 600,
   'max_depth': None,
   'min_samples_leaf': 2,
   'max_features': 'sqrt',
   'random_state': 42},
  'XGBoost': {'n_estimators': 500,
   'learning_rate': 0.08,
   'max_depth': 5,